In [1]:
# ====================================================================
# 🔫 SCRIPT ULTIME : "SNIPER ELITE" (OBJECTIF 0.7700)
# Stratégie : Twin XGBoost + Recall Injection + Nano-Thresholding
# ====================================================================

import pandas as pd
import numpy as np
import warnings

# On ignore les warnings futurs pour avoir une sortie propre
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# --------------------------------------------------------------------
# 1. CHARGEMENT
# --------------------------------------------------------------------
print("⏳ Chargement des données...")
# Assure-toi que les fichiers sont bien dans le même dossier
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

# --------------------------------------------------------------------
# 2. FEATURE ENGINEERING (La formule gagnante)
# --------------------------------------------------------------------
def feature_engineering(df):
    df_eng = df.copy()
    
    # 1. Interaction Âge x Pages (L'intérêt cumulé)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    
    # 2. Utilisateur Actif (Seuil psychologique)
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    
    # 3. Vitesse de consommation (Pages par année d'âge)
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    
    return df_eng

# Application
X = feature_engineering(train_df.drop('converted', axis=1))
y = train_df['converted']
X_test = feature_engineering(test_df)

# --------------------------------------------------------------------
# 3. PREPROCESSING
# --------------------------------------------------------------------
numeric_features = [
    'age', 
    'total_pages_visited', 
    'interaction_age_pages', 
    'pages_per_age'
]
categorical_features = ['country', 'source']

preprocessor = ColumnTransformer(
    transformers=[
        # StandardScaler est crucial pour la Régression Logistique
        ('num', StandardScaler(), numeric_features),
        # OneHot pour les arbres et la LogReg
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

# --------------------------------------------------------------------
# 4. L'ESCOUADE DE MODÈLES (TWIN STRATEGY)
# --------------------------------------------------------------------

# JUMEAU 1 : Le XGBoost classique (Seed 42)
clf_xgb_1 = XGBClassifier(
    n_estimators=350,       # Un peu plus d'arbres pour la robustesse
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.9, 
    colsample_bytree=0.9, 
    eval_metric='logloss', 
    random_state=42,
    n_jobs=1
)

# JUMEAU 2 : Le XGBoost de renfort (Seed 2025)
# Il va corriger les micro-erreurs du premier
clf_xgb_2 = XGBClassifier(
    n_estimators=350, 
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.9, 
    colsample_bytree=0.9, 
    eval_metric='logloss', 
    random_state=2025,      # Graine différente = Arbres différents
    n_jobs=1
)

# Le LightGBM (Vitesse et Précision)
clf_lgbm = LGBMClassifier(
    n_estimators=350, 
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.9, 
    colsample_bytree=0.9, 
    verbose=-1, 
    random_state=42,
    n_jobs=1
)

# Le Gradient Boosting Classique (Sécurité Sklearn)
clf_gb = GradientBoostingClassifier(
    n_estimators=350, 
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.9, 
    random_state=42
)

# L'ARME SECRÈTE : Logistic Regression "Recall-First"
# Elle force le modèle à considérer les cas douteux comme positifs (class_weight 1:80)
clf_log_recall = LogisticRegression(
    max_iter=1000, 
    class_weight={0: 1, 1: 80}, 
    solver='lbfgs', 
    n_jobs=1
)

# --------------------------------------------------------------------
# 5. VOTE PONDÉRÉ (5 EXPERTS)
# --------------------------------------------------------------------
print("🗳️ Constitution du Vote Élite...")
voting_model = VotingClassifier(
    estimators=[
        ('xgb1', clf_xgb_1), 
        ('xgb2', clf_xgb_2), 
        ('lgbm', clf_lgbm), 
        ('gb', clf_gb), 
        ('log_recall', clf_log_recall)
    ],
    voting='soft',
    # Stratégie de poids : 
    # - On divise la puissance XGB en deux (0.7 + 0.7 = 1.4 total)
    # - LGBM reste fort (1.2)
    # - LogReg garde son rôle d'influenceur subtil (0.5)
    weights=[0.7, 0.7, 1.2, 0.7, 0.5], 
    n_jobs=-1
)

# --------------------------------------------------------------------
# 6. SCAN NANOMÉTRIQUE DU SEUIL
# --------------------------------------------------------------------
print("🔍 Scan de précision du seuil (CV 10 Folds)...")
cv_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_model)
])

# On utilise 10 folds pour une estimation ultra-stable du 0.77
y_probas_cv = cross_val_predict(
    cv_pipeline, 
    X, 
    y, 
    cv=20, 
    method='predict_proba', 
    n_jobs=-1
)[:, 1]

# Scan très fin autour de la zone gagnante (0.45 - 0.55)
thresholds = np.arange(0.4500, 0.5500, 0.0005) 
best_f1 = 0
best_thresh = 0.5

for t in thresholds:
    y_pred_t = (y_probas_cv > t).astype(int)
    score = f1_score(y, y_pred_t)
    if score > best_f1:
        best_f1 = score
        best_thresh = t

print(f"\n💎 F1-SCORE ÉLITE ESTIMÉ : {best_f1:.5f}")
print(f"🎯 SEUIL MICROMÉTRIQUE : {best_thresh:.4f}")

# --------------------------------------------------------------------
# 7. SOUMISSION FINALE
# --------------------------------------------------------------------
# On génère le fichier quoi qu'il arrive, c'est ta meilleure version.
print("\n🚀 Entraînement final sur tout le dataset...")
cv_pipeline.fit(X, y)

print("🔮 Prédiction sur le Test Set...")
test_probas = cv_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probas > best_thresh).astype(int)

filename = 'divination_v1_fast_predictions.csv'
sub = pd.DataFrame({'converted': test_preds})
sub.to_csv(filename, index=False)

print(f"✅ TERMINÉ. Fichier '{filename}' prêt.")
print("👉 C'est le moment de vérité. Soumets ce fichier.")

⏳ Chargement des données...
🗳️ Constitution du Vote Élite...
🔍 Scan de précision du seuil (CV 10 Folds)...

💎 F1-SCORE ÉLITE ESTIMÉ : 0.77017
🎯 SEUIL MICROMÉTRIQUE : 0.4820

🚀 Entraînement final sur tout le dataset...
🔮 Prédiction sur le Test Set...
✅ TERMINÉ. Fichier 'divination_v1_fast_predictions.csv' prêt.
👉 C'est le moment de vérité. Soumets ce fichier.
